# Block 3 — Assignment 

**Group G — Team members**

- Marie Lacroix
- Chloé Baruselli
- Bertille Delloye
- Enéa Drezet--Marçot

In [ ]:
import matplotlib.pylab as plt 
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge

# 1. Cost functions

Lets focus on linear regression of the form 

$\mathbf{y} \approx f(\mathbf{X}) = \mathbf{X}\mathbf{w_1} + \mathbf{w_0}.$


#### 1.1 What are the rows of $\mathbf{X}$?

Each row represents one observation (training sample). If there are $m$ observations and $p$ features, row $i$ contains the $p$ feature values measured for observation $i$.

#### 1.2 What are the columns of $\mathbf{X}$?

Each column represents one input feature (predictor variable) measured across all $m$ observations. There are $p$ columns for $p$ features.

Often, we write the equation above as

$\mathbf{y} \approx \mathbf{\tilde{X}}\mathbf{w}$

#### 1.3 How does $\mathbf{\tilde{X}}$ look like in this case (i.e., how does the shape of the matrix change compared to $\mathbf{X}$)?

We add a column of ones to incorporate the intercept $w_0$ into the weights:
$$\tilde X=\begin{bmatrix}1&x_{11}&\cdots&x_{1p}\\\vdots&\vdots&&\vdots\\1&x_{m1}&\cdots&x_{mp}\end{bmatrix},\qquad w=\begin{bmatrix}w_0\\w_1\\\vdots\\w_p\end{bmatrix}.$$
Consequently, the design matrix changes from shape $(m,p)$ to $(m,p+1)$, and $\hat y=\tilde Xw$. 

For machine learning, we need a cost function. Two common choices are the mean-squared error (MSE, $\mathcal{L}_2$), and the mean-absolute error (MAE, $\mathcal{L}_1$)

\begin{align}
    \mathcal{L}_2 &=& \frac{1}{N} \sum_{i=1}^N \left(y_i - f(x_i) \right)^2 \\
    \mathcal{L}_1 &=& \frac{1}{N} \sum_{i=1}^N \left|y_i - f(x_i) \right| 
\end{align}

#### 1.4 In the Jupyter notebook, write a Python function that computes these two cost functions given an error term $\boldsymbol{\epsilon} = \mathbf{y} - \mathbf{\tilde{X}}\mathbf{w}$

In [ ]:
def mean_squared_error(error_vector):
    mean_squared_error = np.mean(np.asarray(error_vector) ** 2)
    return mean_squared_error


In [ ]:
def mean_absolute_error(error_vector):
    mean_absolute_error = np.mean(np.abs(np.asarray(error_vector)))
    return mean_absolute_error

Your code should run as follows

```python
mean_squared_error(np.array([0,0,0]))
> returns 0
```

```python
mean_squared_error(np.array([1,1,1]))
> returns 1
```

#### 1.5 What is the shape of these cost functions as a function of the error

In [ ]:
x_axis = np.linspace(-10,10,100) # change as you wish for your plot
y_mae = [mean_absolute_error(x) for x in x_axis]
y_mse = [mean_squared_error(x) for x in x_axis]

In [ ]:
plt.plot(x_axis, y_mse, label='MSE')
plt.plot(x_axis, y_mae, label='MAE')
plt.xlabel('error term')
plt.ylabel('cost function')
plt.title('Loss functions for a scalar residual')
plt.grid(alpha=0.3)
plt.legend()
plt.show()

The plot of the mean squared error as a function of the error has a shape of a U parabola, increasing quadratically with the error. The one of the mean absolute error has a shape of a V function, increasing linearly with the absolute error.
Both functions have their minimum at $\epsilon=0$.

#### 1.6  Are both loss functions differentiable for all $\boldsymbol{\epsilon}$? What implications does this have for gradient based optimization like gradient descent?

For a scalar error $\varepsilon$, $L_{\rm MSE}=\varepsilon^2$, with derivative $2\varepsilon$ everywhere, including zero. $L_{\rm MAE}=|\varepsilon|$ has derivative $-1$ for $\varepsilon<0$ and $+1$ for $\varepsilon>0$, but is not differentiable at $\varepsilon=0$ (there is a sharp corner). For a vector of $m$ errors, the MSE gradient with respect to the error vector is $2\varepsilon/m$; the MAE gradient away from zero is $\operatorname{sign}(\varepsilon)/m$ componentwise. Standard gradient descent has a well-defined smooth gradient for MSE. For MAE, one can use subgradient methods (at zero, choose any subgradient in $[-1,1]/m$), or a smooth approximation.

#### 1.7 Which loss function is more sensitive to outliers and why?

MSE is more sensitive to outliers. A residual of magnitude $r$ contributes $r^2$ to MSE but only $|r|$ to MAE. Large errors therefore receive disproportionately larger penalties and gradient magnitudes under MSE.

# 2. Regularization

Assume that the columns of $\mathbf{X}$ are linearly independent.
As a refresher of linear algebra, recall when the linear system $\mathbf{X}\mathbf{w} = \mathbf{y}$ has

#### 2.1 One unique solution

A unique solution exists when $\operatorname{rank}(X)=\operatorname{rank}([X\mid y])=p$, the number of unknown coefficients (full column rank and a consistent system). Under the stated assumption of linearly independent columns this holds if and only if $y\in\operatorname{Col}(X)$. Necessarily $m\ge p$.

#### 2.2 No solution

There is no exact solution when $\operatorname{rank}(X)<\operatorname{rank}([X\mid y])$: $y$ does not lie in the column space of $X$. This can happen with independent columns when $m>p$, since an overdetermined system may be inconsistent.

#### 2.3 An infinite number of solutions

Infinitely many solutions exist if $\operatorname{rank}(X)=\operatorname{rank}([X\mid y])<p$: the system is consistent but there are free parameters in the null space. This cannot happen if the columns of $X$ are linearly independent, because then $\operatorname{rank}(X)=p$. It can happen, for example, when there are more columns than rows and the columns are dependent.

#### 2.4 Give a geometrical interpretation of the matrix Rank (10 bonus point if you use the Manim package)

The rank is the number of linearly independent columns (equivalently rows) of a matrix. Geometrically it is the dimension of the space spanned by its columns, i.e. the dimension of the image of the transformation $x\mapsto Xx$. Rank 1 means outputs lie on a line; rank 2 means they span a plane; rank 3 means they span a 3D space. Dependent columns do not introduce new directions. For example, the columns $(1,0)^T$ and $(2,0)^T$ have rank 1, while $(1,0)^T$ and $(0,1)^T$ have rank 2.

The figure below shows how three $2\times2$ matrices transform the unit square: rank 2 maps it to a parallelogram (a 2D region), rank 1 squashes it onto a line, and rank 0 collapses it to a point.

In [ ]:
mats = {
    "rank 2:  [[1, 0.5], [0.3, 1]]": np.array([[1, 0.5], [0.3, 1]]),
    "rank 1:  [[1, 2], [0.5, 1]]":   np.array([[1, 2], [0.5, 1]]),
    "rank 0:  [[0, 0], [0, 0]]":     np.zeros((2, 2)),
}
square = np.array([[0, 1, 1, 0, 0], [0, 0, 1, 1, 0]]) 

fig, axes = plt.subplots(1, 3, figsize=(12, 3.8), sharex=True, sharey=True)
for ax, (title, M) in zip(axes, mats.items()):
    ax.plot(*square, 'k--', lw=1, label='unit square')
    img = M @ square
    ax.fill(*img, alpha=0.35, color='tab:blue')
    ax.plot(*img, color='tab:blue', lw=2, label='image')
    ax.set_title(f"{title}\nrank = {np.linalg.matrix_rank(M)}", fontsize=10)
    ax.set_aspect('equal'); ax.grid(alpha=0.3)
axes[0].legend(loc='upper left', fontsize=8)
plt.tight_layout(); plt.show()

#### 2.5 In general, why can't we solve the linear equation using $\mathbf{y} = \tilde{\mathbf{X}}^{-1}\textbf{w}$? (1 point)

The correct formal rearrangement would be $w=\tilde X^{-1}y$, not $y=\tilde X^{-1}w$. But $\tilde X$ is generally rectangular ($m$ samples versus $p+1$ coefficients), and a matrix inverse exists only for a square, nonsingular matrix. Even for square matrices, dependent columns cause singularity; noise also makes exact fitting often impossible. Hence we normally solve a least-squares problem instead.

#### 2.6 Differentiate above formula step by step and show what will we have?

Let $A=\tilde X$. Expand the squared residual norm:
$$J(w)=(y-Aw)^T(y-Aw)=y^Ty-y^TAw-w^TA^Ty+w^TA^TAw.$$
Since $y^TAw$ is a scalar equal to $w^TA^Ty$,
$$J(w)=y^Ty-2w^TA^Ty+w^TA^TAw.$$
Differentiate with respect to $w$:
$$\nabla_w J=0-2A^Ty+2A^TAw=\boxed{2A^T(Aw-y)}.$$
Here $A^TA$ is symmetric; that is why $\nabla_w(w^TA^TAw)=2A^TAw$.

#### 2.7 if we want $\| y - \tilde{X} w \|_2^2$ to be minimum, what should the derivative be equal to?

At a differentiable unconstrained minimum, the gradient must vanish:
$$\nabla_w J=2\tilde X^T(\tilde Xw-y)=0.$$
Thus the normal equations are $\tilde X^T\tilde Xw=\tilde X^Ty$. If $\tilde X$ has full column rank, $w=(\tilde X^T\tilde X)^{-1}\tilde X^Ty$.

#### 2.8 What is the Hat matrix and what does its diagonal values correspond to?

The hat matrix maps the observed response $y$ onto its ordinary least-squares fitted values:
$$\hat y=Hy,\qquad H=\tilde X(\tilde X^T\tilde X)^{-1}\tilde X^T,$$
assuming full column rank (otherwise use the pseudoinverse). Its diagonal entries $h_{ii}$ are the leverages: they quantify how strongly observation $i$ can influence its own fitted value based on its position in feature space. High leverage indicates an unusual predictor combination, not necessarily a large residual. For the full-rank model $\sum_i h_{ii}=p+1$ when the intercept is included.

#### 2.9 What is Willson’s plot and how does it help in outlier detection?

A Williams plot plots standardized or studentized prediction residuals on the vertical axis versus each observation's leverage $h_{ii}$ on the horizontal axis. Large positive or negative residuals (often outside $\pm3$) indicate possible response outliers; high leverages (often compared with a heuristic warning leverage $h^*=3q/m$, where $q$ is the number of fitted parameters including the intercept) indicate observations unusual in feature space. Points with both high leverage and large residual may be especially influential. Cutoffs are rules of thumb, not universal statistical tests.

#### 2.10 What happens if some columns are linearly dependent? What is the connection to feature selection?

Dependent columns make $\tilde X^T\tilde X$ singular, so its ordinary inverse does not exist and the individual least-squares coefficients are not uniquely identifiable. If the observations can be fitted exactly, infinitely many coefficient vectors may give the same fitted response; otherwise there are still multiple least-squares minimizers. Feature selection can remove redundant or highly correlated predictors, reducing collinearity, improving interpretability and potentially increasing numerical stability. Alternatively, use a pseudoinverse or regularization.

#### 2.11 What will be the new cost function after adding the regularization term?

Adding $L_2$ (ridge) regularization gives
$$J_\lambda(w)=\lVert y-\tilde Xw\rVert_2^2+\lambda\lVert w\rVert_2^2,\qquad\lambda\ge0.$$
This formulation penalizes **all** entries of $w$; in many practical ridge models the intercept is excluded from the penalty, in which case replace $\lVert w\rVert_2^2$ by $\sum_{j=1}^p w_j^2$.

#### 2.12 What will be the new 𝑤 when we differentiate the new cost function and set it to zero.

Write $A=\tilde X$. The derivative is
$$\nabla_w J_\lambda=-2A^Ty+2A^TAw+2\lambda w.$$
Set it to zero:
$$(A^TA+\lambda I)w=A^Ty.$$
Therefore, for $\lambda>0$,
$$w_\lambda=(\tilde X^T\tilde X+\lambda I)^{-1}\tilde X^Ty.$$
If the intercept is unpenalized, replace $\lambda I$ with $\lambda\,\mathrm{diag}(0,1,\dots,1)$ and note that unconditional invertibility then requires additional assumptions.

#### 2.13 Prove that the part of Hat matrix where we want to take the inverse from is always reversible after we introduce the regularization term.

For any nonzero vector $v$ and any strictly positive $\lambda$,
$$v^T(\tilde X^T\tilde X+\lambda I)v=\lVert\tilde Xv\rVert_2^2+\lambda\lVert v\rVert_2^2>0.$$
Thus $\tilde X^T\tilde X+\lambda I$ is symmetric positive definite, so all its eigenvalues are strictly positive and it is invertible, even if $\tilde X$ has dependent columns. The statement does not hold for $\lambda=0$ in general; it assumes all coefficients are penalized.

#### 2.14  What is the shape of the parabola as a function of $a$?

In [ ]:
def parabola(x, a = 1): 
    return a * x ** 2

In [ ]:
x_axis_parabola = np.linspace(-10, 10, 100)

In [ ]:
plt.figure(figsize=(7, 4))
for a in [-2, -0.5, 0, 0.5, 1, 2]:
    plt.plot(x_axis_parabola, parabola(x_axis_parabola, a), label=f'a = {a}')
plt.ylim(-30, 30)
plt.xlabel('x')
plt.ylabel('a x²')
plt.legend(ncol=2)
plt.grid(alpha=0.3)
plt.show()

For $f(x)=ax^2$: if $a>0$, the parabola opens upward; if $a<0$, it opens downward; if $a=0$, the graph is the horizontal line $y=0$. Larger $|a|$ makes it narrower/steeper; smaller nonzero $|a|$ makes it wider/flatter. Its vertex remains at $(0,0)$ and it is symmetric about the $y$-axis.

#### 2.15 Plot the approximation to the function for different order polynomials ($N \in \{1, 2, 16\}$) and with different regularization strength ($\lambda \in \{0, 10^{-3}, 10^{-2}, 1\}$). What do you observe 

The 3-by-4 panel plot below uses **the same noisy samples** in all 12 panels for a fair comparison. $\lambda=0$ means ordinary least squares; the others use ridge regression.

In [ ]:
def true_function(X):
    return np.cos(1.5 * np.pi * X)

In [ ]:
X_test = np.linspace(0, 1, 100) # some grid for us on the x axis

In [ ]:
n_samples = 10
degrees = [1, 2, 16]
rng = np.random.default_rng(42) 
X = np.sort(rng.uniform(0, 1, n_samples))
y = true_function(X) + rng.normal(0, 0.1, n_samples)


In [ ]:
plt.scatter(X, y, label='noisy samples')
plt.plot(X_test, true_function(X_test), c='r', label='true function')
plt.legend()

The following code will fit a polynomial regression, you need to fill the degree

In [ ]:
polynomial_features = PolynomialFeatures(degree=2, include_bias=False)
linear_regression = LinearRegression()
pipeline = Pipeline([('polynomial_features', polynomial_features), ('linear_regression', linear_regression)])
pipeline.fit(X[:, np.newaxis], y)


To plot the result, you can use the following code

In [ ]:
plt.plot(X_test, pipeline.predict(X_test[:, np.newaxis]), label="model of degree ")
plt.scatter(X, y, label='noisy samples')
plt.plot(X_test, true_function(X_test), c='r', label='true function')
plt.legend()

Next, we can investigate the effect of the regularization parameter $\lambda$ (function parameter `alpha`), For this, you can use the following code 

In [ ]:
polynomial_features = PolynomialFeatures(degree=16, include_bias=False)
ridge_regression = Ridge(alpha=1e-2)
pipeline_ridge = Pipeline([('polynomial_features', polynomial_features), ('ridge_regression', ridge_regression)])
pipeline_ridge.fit(X[:, np.newaxis], y)


For plotting you can reuse the following code

In [ ]:
plt.plot(X_test, pipeline_ridge.predict(X_test[:, np.newaxis]), label="model of degree with regularization strength")
plt.scatter(X, y, label='noisy samples')
plt.plot(X_test, true_function(X_test), c='r', label='true function')
plt.legend()

In [ ]:
from sklearn.exceptions import ConvergenceWarning
import warnings
lambdas = [0, 1e-3, 1e-2, 1]
fig, axes = plt.subplots(len(degrees), len(lambdas), figsize=(15, 10), sharex=True, sharey=True)
for i, degree in enumerate(degrees):
    for j, alpha in enumerate(lambdas):
        estimator = LinearRegression() if alpha == 0 else Ridge(alpha=alpha)
        model = Pipeline([('polynomial_features', PolynomialFeatures(degree=degree, include_bias=False)),
                          ('regression', estimator)])
        with warnings.catch_warnings():
            warnings.simplefilter('ignore', category=ConvergenceWarning)
            model.fit(X[:, None], y)
        ax = axes[i, j]
        ax.scatter(X, y, color='black', s=20, label='noisy data')
        ax.plot(X_test, true_function(X_test), color='tab:green', ls='--', label='true function')
        ax.plot(X_test, model.predict(X_test[:, None]), color='tab:blue', label='fit')
        ax.set_title(f'Degree {degree}, λ = {alpha:g}')
        ax.grid(alpha=0.2)
        if i == len(degrees)-1: ax.set_xlabel('x')
        if j == 0: ax.set_ylabel('y')
axes[0, 0].legend(loc='best', fontsize=8)
plt.tight_layout()
plt.show()


Degree 1 gives a straight line and systematically underfits the curved cosine, regardless of regularization. Degree 2 captures some curvature but remains too simple to reproduce the whole cosine. Degree 16 without regularization can follow the training noise and produce strong oscillations/unstable predictions between samples (overfitting). With moderate ridge regularization ($10^{-3}$ or $10^{-2}$), coefficient magnitudes are penalized and the high-degree prediction typically becomes smoother and less extreme. With strong regularization ($1$), the model is often excessively flattened and underfits.

This illustrates Occam’s razor: prefer the simplest sufficiently accurate model rather than extra complexity that fits noise. Ridge does not literally penalize curvature; it penalizes coefficients and thereby typically suppresses unstable high-order behavior. The exact shapes depend on the random training sample and on feature scaling.


#### 2.16 What do you observe if you change the number of samples from the function?

With more training samples, the fitted curve is usually better constrained by the data and becomes less sensitive to any individual noisy point. This is especially important for high-degree models, which can interpolate a small sample and oscillate badly between observations. With fewer samples, overfitting and variance become more likely; regularization can help. Results also depend on which $x$ values were sampled and on noise, so improvements are not guaranteed for every particular random draw.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.7), sharex=True, sharey=True)
for ax, count in zip(axes, [10, 30, 100]):
    sample_rng = np.random.default_rng(100 + count)
    x_s = np.sort(sample_rng.uniform(0, 1, count))
    y_s = true_function(x_s) + sample_rng.normal(0, 0.1, count)
    model = Pipeline([('poly', PolynomialFeatures(degree=16, include_bias=False)),
                      ('ridge', Ridge(alpha=1e-3))])
    model.fit(x_s[:, None], y_s)
    ax.plot(X_test, true_function(X_test), '--', label='true function')
    ax.plot(X_test, model.predict(X_test[:, None]), label='fit')
    ax.scatter(x_s, y_s, s=9, color='black', label='observations')
    ax.set_title(f'{count} samples, degree 16, λ=0.001')
    ax.grid(alpha=0.2)
    ax.set_xlabel('x')
axes[0].set_ylabel('y')
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

#### 2.17 Why do we need a test set in machine learning?

A test set is held out from model fitting to estimate performance on unseen data. It gives a more realistic measure of generalization and helps reveal overfitting; evaluating only on training data usually produces an optimistically biased performance estimate.

#### 2.18 If we need to optimize hyperparameters, do we use the test set to select the best hyperparameters? (1 point)

No. Hyperparameters (such as polynomial degree or ridge strength $\lambda$) should be chosen using a validation set or cross-validation within the training data. Only after choosing the model should we evaluate it on the untouched test set once for final generalization assessment. Selecting hyperparameters using test performance leaks information and biases the result.